# 🧑‍⚖️ 11 — Calibrating Judges with Human Labels

Notebook 10 built four judges for the store's house rules and saved every verdict. None of them is
trusted yet: a judge is a model, and a model can be confidently wrong in the same direction every
time. This notebook puts a sample of those verdicts in front of a person, measures how often the
judge agrees with them, checks the judge for three known biases, and runs a second judge model
beside it. The output is one table that says which judges may gate a release.

## Learning Objectives
In this notebook, you will learn:
1. **Sampling for calibration** - a stratified sample per judge, so passes and fails are both checked
2. **Human labels without anchoring** - a LangSmith annotation queue (or a JSON file) where the person never sees the judge's verdict
3. **Agreement in plain Python** - confusion matrix, Cohen's kappa, true-positive and true-negative rates, and the 0.6 bar
4. **Bias checks** - position, verbosity and self-preference, each measured on this course's own judges
5. **A reference judge and a trust table** - `gpt-4o` beside the Qwen judge, and `data/judge_trust.json` for the release gate

## Where this fits
Notebook 10 wrote `data/custom_judge_scores.json`: pass/fail verdicts with reasons. This notebook
turns those verdicts into a trust table. Notebook 12 runs the statistics on every score, and
notebook 13's release gate counts a judge only if this table marks it trusted.

## Prerequisites
- Notebook 10 (`data/custom_judge_scores.json`), notebook 03 (`data/golden_set.json`) and notebook 04
  (`data/golden_runs.json`, the baseline run the verdicts are about)
- `OPENROUTER_API_KEY` (the Qwen judge, the `gpt-4o` reference judge and the DeepSeek app model as a
  judge) and a LangSmith key (`LANGSMITH_API_KEY`, or `LANGCHAIN_API_KEY`, which the SDK also reads)
  in the `.env` file at the repository root
- **A person with 1–2 hours** to label about 200 items. Parts 3 and 7 need those labels; without
  them they say so and stop. No label in this notebook is ever made up

---
## 🗺️ The idea

```
 data/custom_judge_scores.json (notebook 10)
    │
    ├── 1. sample ≤ 60 verdicts per judge, passes and fails both
    ├── 2. a person labels the same items pass/fail, without seeing the verdict
    │        LangSmith annotation queue ─┐
    │        or data/human_labels_todo.json ┴─→ data/human_labels.json
    ├── 3. agreement: confusion matrix, kappa, TPR, TNR per judge
    ├── 4. bias: position (free), verbosity, self-preference
    ├── 5. gpt-4o as a reference judge on the same items
    └── 7. data/judge_trust.json → notebook 13's release gate
```

Two rules hold throughout. The person labels **blind**: a labeller who sees "the judge said pass"
agrees with it more often, which inflates the very number being measured. And a judge with no
human labels is **not trusted**, however reasonable its verdicts look.

---
## ⚙️ Part 0: Setup

The setup loads notebook 10's verdicts and the baseline run they judged, and builds the Qwen judge
exactly as notebook 10 did. Nothing is traced automatically: the only LangSmith writes are the
labelling runs in Part 2, made on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: keys, telemetry, the judge model and notebook 10's verdicts
# ============================================================================
import os
import tomllib
import json
import random
from collections import Counter, defaultdict

from dotenv import load_dotenv

load_dotenv(os.path.join("..", "..", ".env"))
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"   # no DeepEval usage telemetry
os.environ["LANGSMITH_TRACING"] = "false"         # Part 2 writes its runs explicitly; nothing else is traced

from deepeval.models import GPTModel

# Every model and provider comes from models.toml: change them there, not in the notebooks.
with open("models.toml", "rb") as f:
    MODELS_CONFIG = tomllib.load(f)

def model_config(role):
    """One role from models.toml, with its provider's base URL, key variable and key filled in."""
    model = dict(MODELS_CONFIG["models"][role])
    provider = MODELS_CONFIG["providers"][model["provider"]]
    model["base_url"] = provider["base_url"]
    model["api_key_env"] = provider["api_key_env"]
    model["api_key"] = os.getenv(provider["api_key_env"]) or "missing"   # never fall back to another provider's key
    model.setdefault("extra_body", {})
    return model

print("✅ Config loaded; roles:", list(MODELS_CONFIG["models"]))

In [ ]:
# ============================================================================
# THE PRIMARY JUDGE: Qwen3, built exactly as in notebook 10
# ============================================================================
JUDGE = model_config("judge")
JUDGE_MODEL_NAME = JUDGE["name"]
JUDGE_MODEL = GPTModel(model=JUDGE_MODEL_NAME, base_url=JUDGE["base_url"], api_key=JUDGE["api_key"],
                       generation_kwargs={"extra_body": JUDGE["extra_body"]}, temperature=0)
JUDGE_MODEL.model_data.supports_structured_outputs = JUDGE.get("structured_outputs", False)

# --- The inputs, each written by an earlier notebook
inputs = {"custom_judge_scores.json": "10", "golden_set.json": "03", "golden_runs.json": "04"}
for file_name, notebook in inputs.items():
    if not os.path.exists(os.path.join("data", file_name)):
        raise RuntimeError(f"No data/{file_name}. Run notebook {notebook} first: it writes it.")

with open(os.path.join("data", "custom_judge_scores.json"), encoding="utf-8") as f:
    custom_judge_scores = json.load(f)
with open(os.path.join("data", "golden_set.json"), encoding="utf-8") as f:
    golden_set = json.load(f)
with open(os.path.join("data", "golden_runs.json"), encoding="utf-8") as f:
    golden_runs = json.load(f)
cases_by_id = {case["id"]: case for case in golden_set}
runs_by_id = {run["run_id"]: run for run in golden_runs}

print("✅ Judge model:", JUDGE_MODEL_NAME, "| verdicts written by:", custom_judge_scores["judge_model"])
print(f"📂 {len(custom_judge_scores['rubric'])} rubric, {len(custom_judge_scores['refund_dag'])} tree, "
      f"{len(custom_judge_scores['pairwise'])} pairwise verdicts")

In [ ]:
# ============================================================================
# WHAT THE AGENT DID, AS PLAIN FACTS (copied from notebook 10)
# ============================================================================
# Written by code from the run record, never by a model. The judges read these facts, and so does
# the person labelling: both must judge the same evidence.
def actions_text(run):
    tools = [f"{call['name']}({', '.join(f'{k}={v}' for k, v in call['args'].items())})"
             for call in run["tool_calls"]] or ["none"]
    refunds = [f"{refund['order_id']} (${refund['amount']})" for refund in run["refunds_issued"]] or ["none"]
    tickets = [f"{ticket['ticket_id']} ({ticket['priority']})" for ticket in run["tickets_created"]] or ["none"]
    return (f"What the assistant did. Tools called: {'; '.join(tools)}. Refunds issued: {', '.join(refunds)}. "
            f"Tickets created for a human agent: {', '.join(tickets)}.")

# The exact check notebook 10 set beside the refund-flow judge: was the decision right?
def end_state_correct(run):
    expected = cases_by_id[run["case_id"]]["expected_end_state"]
    refunded = sorted(refund["order_id"] for refund in run["refunds_issued"])
    return refunded == sorted(expected["refund_order_ids"]) and bool(run["tickets_created"]) == expected["ticket_created"]

print("📋", actions_text(golden_runs[0]))

### 0.1 📏 The judges under test, copied from notebook 10

Calibration measures one exact judge: its model, its prompt and its steps. Change a word in a
rubric and the measured agreement no longer applies. So the four judges below are notebook 10's
definitions copied word for word; only the `model` is a parameter, because Parts 4 and 5 put the
same rubrics in front of other models.

In [ ]:
# ============================================================================
# THE THREE RUBRIC JUDGES (verbatim from notebook 10; only the model varies)
# ============================================================================
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

TONE_STEPS = [
    "Read the customer's message and the assistant's reply.",
    "The reply must be polite and calm, even when the customer is rude, upset or demanding.",
    "It must not blame the customer, lecture them, or sound sarcastic or cold.",
    "It must be plain and to the point: no filler, and at most one apology.",
    "Pass only if every point holds. A correct reply that is curt or long-winded fails.",
]
POLICY_WORDING_STEPS = [
    "The retrieval context is the support articles the assistant was given.",
    "Find every store rule the reply states: a time window, a money limit, a fee, a condition.",
    "Each rule must use the articles' exact numbers and conditions ('30 days from delivery', not 'about a month').",
    "A rule the articles state firmly must not be softened ('usually', 'you may be able to'), and no "
    "exception the articles don't give may be added.",
    "Each stated rule must be followed by the chunk ID it came from, in square brackets.",
    "Pass only if every stated rule meets all three points. A reply that states no rule passes.",
]
ESCALATION_STEPS = [
    "The context says what the assistant actually did, including any ticket created for a human agent.",
    "If a ticket was created, the reply must say that a human agent will follow up by email, within "
    "1 business day (4 hours for a safety incident).",
    "If no ticket was created, the reply must not claim that one was, or that a person will be in touch.",
    "The reply must not promise what the human will decide (for example 'your refund will be approved').",
    "It must not ask the customer to repeat details they already gave.",
    "Pass only if every point holds.",
]
print(f"✅ Rubric steps: tone {len(TONE_STEPS)}, policy wording {len(POLICY_WORDING_STEPS)}, "
      f"escalation {len(ESCALATION_STEPS)}")

In [ ]:
# ============================================================================
# BUILD THE RUBRIC JUDGES FOR ANY MODEL
# ============================================================================
def rubric_judges(model):
    tone = GEval(name="Tone", evaluation_steps=TONE_STEPS,
                 evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
                 model=model, strict_mode=True, async_mode=False)
    policy = GEval(name="Policy wording", evaluation_steps=POLICY_WORDING_STEPS,
                   evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                                      SingleTurnParams.RETRIEVAL_CONTEXT],
                   model=model, strict_mode=True, async_mode=False)
    escalation = GEval(name="Escalation phrasing", evaluation_steps=ESCALATION_STEPS,
                       evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                                          SingleTurnParams.CONTEXT],
                       model=model, strict_mode=True, async_mode=False)
    return {"tone": tone, "policy_wording": policy, "escalation_phrasing": escalation}

print("✅ Rubric judges for the Qwen model:", list(rubric_judges(JUDGE_MODEL)))

In [ ]:
# ============================================================================
# THE REFUND-FLOW DECISION TREE (verbatim from notebook 10)
# ============================================================================
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph, BinaryJudgementNode, NonBinaryJudgementNode

def reply_check(criteria):
    # A leaf question about the reply: yes scores 10, no scores 0.
    node = BinaryJudgementNode(criteria=criteria,
                               evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT])
    node.add_verdict(True, score=10)
    node.add_verdict(False, score=0)
    return node

verified = BinaryJudgementNode(
    criteria="Does the customer's message give BOTH an order ID (like PC-10003) AND an email address?",
    evaluation_params=[SingleTurnParams.INPUT],
)
verified.add_verdict(False, then=reply_check(
    "The customer did not give both an order ID and an email. Does the reply ask for what is missing "
    "(or hand over to a human) WITHOUT issuing, confirming or promising a refund?"))
outcome = NonBinaryJudgementNode(
    criteria="According to the context, which of these happened?",
    evaluation_params=[SingleTurnParams.CONTEXT],
)
print("✅ Tree root and branch questions ready")

In [ ]:
# ============================================================================
# THE TREE'S LEAVES, AND A REFUND-FLOW JUDGE FOR ANY MODEL
# ============================================================================
outcome.add_verdict("a refund was issued", then=reply_check(
    "A refund was issued. Does the reply confirm it, with the amount shown in the context, without "
    "promising anything else?"))
outcome.add_verdict("a ticket was created for a human agent", then=reply_check(
    "The request was handed to a human. Does the reply say why (for example over the $500 limit, or "
    "outside the return window), say that a human agent will reply by email, and NOT say a refund was issued?"))
outcome.add_verdict("no refund and no ticket", then=reply_check(
    "Nothing was refunded or handed over. Does the reply name the store rule that prevents the refund "
    "(sale window, return window, not delivered, already refunded, or details that don't match), and "
    "NOT say or imply that a refund was issued?"))
verified.add_verdict(True, then=outcome)
REFUND_DAG = DeepAcyclicGraph(root_nodes=[verified])

def all_judges(model):
    judges = rubric_judges(model)
    judges["refund_flow"] = DAGMetric(name="Refund flow", dag=REFUND_DAG, model=model, async_mode=False)
    return judges

print("✅ Four judges per model:", list(all_judges(JUDGE_MODEL)))

---
## 🎲 Part 1: Sampling for calibration

Labelling every verdict is slow, and unnecessary: kappa from 50–100 items is tight enough to
tell a good judge from a bad one. Two things matter about the sample:

- **Both verdicts appear.** Most replies pass most rules. A random 60 might hold three fails, and
  then nobody learns whether the judge catches failures, which is the part that matters. So the
  sample takes up to half fails, and fills the rest with passes.
- **It is reproducible.** A fixed seed means the same items come back when the notebook runs
  again, so labels already given still line up with their items.

> **Key Insight**: a judge that passes everything scores 90% "accuracy" on a sample that is 90%
> passes. Stratifying by verdict, and reporting kappa instead of accuracy, stops that hiding.

In [ ]:
# ============================================================================
# ONE POOL OF VERDICTS PER JUDGE
# ============================================================================
pools = defaultdict(list)          # judge → list of {key, judge, case_id, run_id, slice, passed, reason}
for verdict in custom_judge_scores["rubric"]:
    pools[verdict["judge"]].append(verdict)
for verdict in custom_judge_scores["refund_dag"]:
    pools["refund_flow"].append(dict(verdict, judge="refund_flow"))

for judge_name, pool in pools.items():
    for verdict in pool:
        verdict["key"] = f"{judge_name}::{verdict['case_id']}"   # one item = one judge on one case
    passed = sum(verdict["passed"] for verdict in pool)
    print(f"🧮 {judge_name:<20} {len(pool):>3} verdicts: {passed} pass, {len(pool) - passed} fail")

In [ ]:
# ============================================================================
# THE STRATIFIED SAMPLE: up to half fails, the rest passes, fixed seed
# ============================================================================
SAMPLE_PER_JUDGE = 60      # within the 50–100 the calibration spec asks for
rng = random.Random(11)    # fixed seed: the same items every run, so labels stay attached to them

sample = []
for judge_name, pool in pools.items():
    fails = sorted((v for v in pool if not v["passed"]), key=lambda v: v["case_id"])
    passes = sorted((v for v in pool if v["passed"]), key=lambda v: v["case_id"])
    rng.shuffle(fails)
    rng.shuffle(passes)
    chosen = fails[:SAMPLE_PER_JUDGE // 2]
    chosen += passes[:SAMPLE_PER_JUDGE - len(chosen)]
    sample.extend(chosen)
    print(f"🎲 {judge_name:<20} sampled {len(chosen):>3} of {len(pool):>3} "
          f"({sum(not v['passed'] for v in chosen)} fails)")
sample_by_key = {item["key"]: item for item in sample}
print(f"📋 {len(sample)} items to label in total")

Reading the counts: the escalation and refund-flow judges only ever saw a few dozen replies (a
rule judged where it doesn't apply would pass for free), so their whole pool is the sample. Fewer
than 50 items still gives a kappa, with a wider margin of error; Part 3 prints the count beside
every kappa so the reader can weigh it.

---
## 🏷️ Part 2: Human labels, without anchoring

The person labels the same thing the judge saw: the rule, the customer's message, the reply, and
the facts the judge was given (the support articles for policy wording, what the assistant did
for escalation and refunds). They do **not** see the judge's verdict or its reason. Each item
gets one answer: does the reply pass this rule?

Two ways to collect the labels, both ending in the same file, `data/human_labels.json`:

- **2.2–2.3 A LangSmith annotation queue** (the course default): items appear in a review UI, the
  person clicks pass or fail, and the labels stay attached to the item in LangSmith.
- **2.4 A JSON file** (no account needed): `data/human_labels_todo.json`, filled in a text editor.

### Key Concepts:
- **Anchoring**: a labeller shown the judge's answer drifts toward it, so agreement looks higher
  than it is
- **Annotation queue**: a LangSmith list of runs waiting for a person's feedback, with
  instructions and a fixed feedback key (here `pass`)

In [ ]:
# ============================================================================
# WHAT THE LABELLER SEES FOR ONE ITEM (and what they don't)
# ============================================================================
RULE_TEXT = {
    "tone": " ".join(TONE_STEPS),
    "policy_wording": " ".join(POLICY_WORDING_STEPS),
    "escalation_phrasing": " ".join(ESCALATION_STEPS),
    "refund_flow": ("Refund replies. If the customer did not give both an order ID and an email, the reply "
                    "asks for what is missing and issues or promises no refund. If a refund was issued, it "
                    "confirms the amount. If the case went to a human, it says why and that a human will reply "
                    "by email, and does not say a refund was issued. If nothing happened, it names the rule "
                    "that prevents the refund."),
}

def labelling_view(item):
    run = runs_by_id[item["run_id"]]
    if item["judge"] == "policy_wording":
        evidence = "\n\n".join(f"[{chunk_id}] {text}" for chunk_id, text in zip(run["retrieved_chunk_ids"], run["contexts"]))
    elif item["judge"] == "tone":
        evidence = "(the tone rule needs only the message and the reply)"
    else:
        evidence = actions_text(run)
    # No "passed", no "reason": the judge's answer never reaches the labeller.
    return {"rule": RULE_TEXT[item["judge"]], "customer_message": run["question"],
            "assistant_reply": run["answer"], "evidence": evidence}

print(json.dumps(labelling_view(sample[0]), indent=2, ensure_ascii=False)[:900])

### 2.1 📥 Send the sample to a LangSmith annotation queue

The baseline traces from notebook 04 have expired (the free plan keeps traces 14 days), so each
item is logged as a fresh, small run in its own project, `end-to-end-eval-calibration`. The run's
inputs are the labelling view above; it has no outputs, so nothing about the verdict is stored.
The cell records which run holds which item in `data/calibration_queue.json`, and skips the upload
if that file already exists, so running it twice never makes a second queue.

⏱️ No model calls. About 240 small runs count toward LangSmith's free 5,000 traces a month.

In [ ]:
# ============================================================================
# CREATE THE FEEDBACK KEY AND THE QUEUE
# ============================================================================
import uuid
from datetime import datetime, timezone

from langsmith import Client

client = Client()   # reads LANGSMITH_API_KEY, or LANGCHAIN_API_KEY
PROJECT = "end-to-end-eval-calibration"
QUEUE_NAME = "course-04-judge-calibration"

# A categorical key: the reviewer picks "pass" (1) or "fail" (0), nothing in between.
if not list(client.list_feedback_configs(feedback_key=["pass"])):
    client.create_feedback_config("pass", feedback_config={
        "type": "categorical", "categories": [{"value": 1, "label": "pass"}, {"value": 0, "label": "fail"}]})

existing_queues = list(client.list_annotation_queues(name=QUEUE_NAME))
if existing_queues:
    queue = existing_queues[0]
else:
    queue = client.create_annotation_queue(
        name=QUEUE_NAME,
        description="Course 04, notebook 11: does each reply pass the rule shown with it?",
        rubric_instructions=("Read the rule, the customer's message, the reply and the evidence. Answer only "
                             "this: does the reply pass the rule? Judge the reply, not the store policy."),
        rubric_items=[{"feedback_key": "pass", "description": "Does the reply pass the rule?", "is_required": True}],
    )
print("✅ Queue ready:", queue.name, queue.id)

In [ ]:
# ============================================================================
# LOG ONE RUN PER ITEM AND ADD THEM TO THE QUEUE
# ============================================================================
QUEUE_FILE = os.path.join("data", "calibration_queue.json")
if os.path.exists(QUEUE_FILE):
    print("✅ Items already in the queue (data/calibration_queue.json); nothing uploaded")
else:
    run_to_item = {}
    for item in sample:
        run_id = uuid.uuid4()
        now = datetime.now(timezone.utc)
        client.create_run(id=run_id, project_name=PROJECT, name=f"label: {item['judge']}", run_type="chain",
                          inputs=labelling_view(item), outputs={}, start_time=now, end_time=now)
        run_to_item[str(run_id)] = item["key"]
    client.flush()   # make sure every run reached LangSmith before the queue points at it
    client.add_runs_to_annotation_queue(queue.id, run_ids=list(run_to_item))
    with open(QUEUE_FILE, "w", encoding="utf-8") as f:
        json.dump({"queue_id": str(queue.id), "runs": run_to_item}, f, indent=2)
    print(f"✅ {len(run_to_item)} items logged and queued → data/calibration_queue.json")

### 2.2 🧑 Label the queue (a person, outside the notebook)

Open LangSmith → **Annotation queues** → `course-04-judge-calibration`. For each item, read the
rule and the evidence, then set `pass` to **pass** or **fail**. Labelling is the only step here a
model can't do: the whole point is a second opinion that isn't a model.

Allow 20–30 seconds an item. Then run the next cell, which is tagged `skip-execution` because it
only means something after a person has labelled. If it runs earlier, it reads zero labels and
writes nothing.

In [ ]:
# ============================================================================
# READ THE HUMAN LABELS BACK → data/human_labels.json
# ============================================================================
LABELS_FILE = os.path.join("data", "human_labels.json")
with open(QUEUE_FILE, encoding="utf-8") as f:
    run_to_item = json.load(f)["runs"]

human_labels = {}
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE, encoding="utf-8") as f:
        human_labels = json.load(f)
new_labels = 0
for feedback in client.list_feedback(run_ids=list(run_to_item), feedback_key=["pass"]):
    labeller = feedback.feedback_source.user_name if feedback.feedback_source else None
    human_labels[run_to_item[str(feedback.run_id)]] = {
        "label": feedback.score == 1, "labeller": labeller or "unknown", "source": "langsmith"}
    new_labels += 1

if human_labels:
    with open(LABELS_FILE, "w", encoding="utf-8") as f:
        json.dump(human_labels, f, indent=2)
print(f"🏷️ {new_labels} labels read from LangSmith; {len(human_labels)} of {len(sample)} items labelled in total")

### 2.3 📝 Or label in a JSON file

Without LangSmith, the next cell writes `data/human_labels_todo.json`: one entry per item with the
same view the queue shows and `"label": null`. Open it in an editor, set each `label` to `true`
(pass) or `false` (fail), put your name in `labeller`, save, and run the cell after it. The cell
never overwrites a todo file that already exists, so labels typed into it are safe.

In [ ]:
# ============================================================================
# WRITE THE LABELLING FILE (once)
# ============================================================================
TODO_FILE = os.path.join("data", "human_labels_todo.json")
if os.path.exists(TODO_FILE):
    print("✅ data/human_labels_todo.json already exists; not overwritten")
else:
    todo = []
    for item in sample:
        todo.append({"key": item["key"], **labelling_view(item), "label": None, "labeller": ""})
    with open(TODO_FILE, "w", encoding="utf-8") as f:
        json.dump(todo, f, indent=2, ensure_ascii=False)
    print(f"📝 {len(todo)} items written to data/human_labels_todo.json: set each label to true or false")

In [ ]:
# ============================================================================
# READ THE FILLED-IN FILE → data/human_labels.json
# ============================================================================
LABELS_FILE = os.path.join("data", "human_labels.json")
human_labels = {}
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE, encoding="utf-8") as f:
        human_labels = json.load(f)
with open(TODO_FILE, encoding="utf-8") as f:
    todo = json.load(f)

from_file = 0
for entry in todo:
    if entry["label"] is True or entry["label"] is False:   # null means "not labelled yet"
        human_labels[entry["key"]] = {"label": entry["label"], "labeller": entry["labeller"] or "unknown",
                                      "source": "file"}
        from_file += 1
if human_labels:
    with open(LABELS_FILE, "w", encoding="utf-8") as f:
        json.dump(human_labels, f, indent=2)
print(f"🏷️ {from_file} labels read from the file; {len(human_labels)} of {len(sample)} items labelled in total")

---
## 🤝 Part 3: Agreement between the judge and the person

Each labelled item gives a pair: the judge's verdict and the person's label. Four counts come out
of those pairs, and every number below is built from them:

|              | person: pass | person: fail |
|---|---|---|
| **judge: pass** | true positive (TP) | false positive (FP) |
| **judge: fail** | false negative (FN) | true negative (TN) |

### Key Concepts:
- **TPR** (true-positive rate) `TP / (TP + FN)`: of the replies a person passes, how many the judge passes
- **TNR** (true-negative rate) `TN / (TN + FP)`: of the replies a person fails, how many the judge
  catches. For a gate this is the number that matters: a low TNR lets bad replies through
- **Cohen's kappa** `(p_o − p_e) / (1 − p_e)`: observed agreement `p_o` corrected for the agreement
  `p_e` two raters would reach by chance with the same pass rates. 0 is chance, 1 is perfect. The
  course bar is **kappa ≥ 0.6** before a judge counts in the release gate

In [ ]:
# ============================================================================
# THE LABELS, IF A PERSON HAS GIVEN THEM
# ============================================================================
LABELS_FILE = os.path.join("data", "human_labels.json")
human_labels = {}
if os.path.exists(LABELS_FILE):
    with open(LABELS_FILE, encoding="utf-8") as f:
        human_labels = json.load(f)

if human_labels:
    sources = Counter(label["source"] for label in human_labels.values())
    print(f"🏷️ {len(human_labels)} human labels ({dict(sources)}) for {len(sample)} sampled items")
else:
    print("⏸️ No data/human_labels.json yet. Label the items (Part 2), then run Part 3 again.\n"
          "   Every cell that needs labels prints this reminder and does nothing else.")

In [ ]:
# ============================================================================
# AGREEMENT FROM PAIRS: confusion counts, TPR, TNR, kappa (standard library only)
# ============================================================================
def agreement(pairs):
    """pairs: list of (judge_says_pass, person_says_pass). Returns the counts and the three rates."""
    tp = sum(1 for judge, person in pairs if judge and person)
    fp = sum(1 for judge, person in pairs if judge and not person)
    fn = sum(1 for judge, person in pairs if not judge and person)
    tn = sum(1 for judge, person in pairs if not judge and not person)
    n = tp + fp + fn + tn
    if n == 0:
        return None
    p_observed = (tp + tn) / n
    # Chance agreement: both say pass by chance, plus both say fail by chance.
    p_chance = ((tp + fp) / n) * ((tp + fn) / n) + ((fn + tn) / n) * ((fp + tn) / n)
    kappa = 1.0 if p_chance == 1 else (p_observed - p_chance) / (1 - p_chance)
    tpr = tp / (tp + fn) if tp + fn else None
    tnr = tn / (tn + fp) if tn + fp else None
    return {"n": n, "tp": tp, "fp": fp, "fn": fn, "tn": tn, "kappa": kappa, "tpr": tpr, "tnr": tnr}

# A check anyone can do by hand: 8 agree-pass, 1 judge-only pass, 1 judge-only fail, 0 agree-fail.
print("🧮 sanity check:", agreement([(True, True)] * 8 + [(True, False), (False, True)]))

Reading the sanity check: the judge and the person agree on 8 of 10 items, yet kappa comes out
slightly *negative*. Both said "pass" almost every time, so they would agree that often by chance;
and the one reply the person failed, the judge passed (TNR 0). This is exactly the judge kappa is
built to expose: high agreement, no ability to catch a failure.

In [ ]:
# ============================================================================
# THE QWEN JUDGE AGAINST THE PERSON, PER JUDGE
# ============================================================================
def fmt(value):
    return "  n/a" if value is None else f"{value:5.2f}"

primary_agreement = {}
if not human_labels:
    print("⏸️ Needs human labels (Part 2).")
else:
    print(f"{'judge':<20}{'n':>4}{'TP':>4}{'FP':>4}{'FN':>4}{'TN':>4}{'kappa':>7}{'TPR':>6}{'TNR':>6}  trusted?")
    for judge_name in pools:
        pairs = [(item["passed"], human_labels[item["key"]]["label"])
                 for item in sample if item["judge"] == judge_name and item["key"] in human_labels]
        result = agreement(pairs)
        if result is None:
            print(f"{judge_name:<20} no labels yet")
            continue
        primary_agreement[judge_name] = result
        trusted = "✅" if result["kappa"] >= 0.6 else "❌"
        print(f"{judge_name:<20}{result['n']:>4}{result['tp']:>4}{result['fp']:>4}{result['fn']:>4}{result['tn']:>4}"
              f"{fmt(result['kappa']):>7}{fmt(result['tpr']):>6}{fmt(result['tnr']):>6}  {trusted}")
    print("Reading: kappa ≥ 0.6 is the bar. Under it, read the disagreements below before using the judge.")

In [ ]:
# ============================================================================
# READ THE DISAGREEMENTS: is the judge wrong, the rubric unclear, or the label wrong?
# ============================================================================
if not human_labels:
    print("⏸️ Needs human labels (Part 2).")
else:
    for judge_name in pools:
        disagreements = [item for item in sample if item["judge"] == judge_name and item["key"] in human_labels
                         and item["passed"] != human_labels[item["key"]]["label"]]
        print(f"\n⚖️ {judge_name}: {len(disagreements)} disagreements")
        for item in disagreements[:3]:
            run = runs_by_id[item["run_id"]]
            person = "pass" if human_labels[item["key"]]["label"] else "fail"
            print(f"  {item['case_id']}: judge {'pass' if item['passed'] else 'fail'}, person {person}")
            print(f"    reply : {run['answer'][:160]}")
            print(f"    judge : {item['reason'][:200]}")

Each disagreement has one of three causes, and each has a different fix:

- **The judge misread the evidence** (its reason quotes something the reply doesn't say): a model
  problem. Try the reference judge in Part 5; if it agrees with the person, the judge model is the issue.
- **The rubric is ambiguous** (the judge's reading is defensible, the person read it another way):
  rewrite the step that both readings hang on. Part 6 covers this.
- **The label is wrong** (on a second read, the judge was right): fix the label. Labels are
  data too, and a second labeller on the disagreements is the cheap way to find these.

---
## 🧭 Part 4: Bias checks

Agreement measures how often a judge is right. A bias check asks whether it is wrong **in a
predictable direction**, triggered by something that shouldn't matter. Three are checked here:

| Bias | What shouldn't matter | How it's measured | Cost |
|---|---|---|---|
| Position | which reply is shown first | notebook 10's pairwise verdicts, both orders | free |
| Verbosity | padding that adds no content | re-judge padded replies, count flips | paid |
| Self-preference | the judge wrote the reply | the app model judges its own replies | paid |

The paid checks keep their verdicts in `data/calibration_verdicts.json`, so running the notebook
again after labelling doesn't pay for them twice.

In [ ]:
# ============================================================================
# A CACHE FOR PAID VERDICTS, AND ONE CALL THAT RETRIES
# ============================================================================
VERDICTS_FILE = os.path.join("data", "calibration_verdicts.json")
calibration_verdicts = {"verbosity": {}, "self_preference": {}, "reference": {}}
if os.path.exists(VERDICTS_FILE):
    with open(VERDICTS_FILE, encoding="utf-8") as f:
        calibration_verdicts = json.load(f)

def save_verdicts():
    with open(VERDICTS_FILE, "w", encoding="utf-8") as f:
        json.dump(calibration_verdicts, f, indent=2, ensure_ascii=False)

def judge_passes(judge, test_case, attempts=3):
    # The Qwen judge sometimes runs into its output limit; a retry usually succeeds.
    for attempt in range(attempts):
        try:
            judge.measure(test_case, _show_indicator=False)
            return bool(judge.success)
        except Exception as error:
            last_error = error
    print(f"⚠️ gave up after {attempts} attempts: {type(last_error).__name__}")
    return None   # no verdict: left out of every count

print("✅ Cached verdicts:", {name: len(saved) for name, saved in calibration_verdicts.items()})

In [ ]:
# ============================================================================
# THE TEST CASE EACH JUDGE NEEDS (the same fields as notebook 10)
# ============================================================================
def test_case_for(judge_name, run, reply=None):
    reply = run["answer"] if reply is None else reply
    if judge_name == "tone":
        return LLMTestCase(input=run["question"], actual_output=reply)
    if judge_name == "policy_wording":
        return LLMTestCase(input=run["question"], actual_output=reply, retrieval_context=run["contexts"])
    return LLMTestCase(input=run["question"], actual_output=reply, context=[actions_text(run)])

print("✅ Test case for", sample[0]["key"], "→", type(test_case_for(sample[0]["judge"], runs_by_id[sample[0]["run_id"]])).__name__)

### 4.1 📐 Position bias (free)

Notebook 10 judged each pair of replies twice, once in each order. An unbiased judge picks the
same reply both times. A **flip** (the winner changes when the order changes) means the order, not
the replies, decided; and if the reply shown first wins far more than half the time, the judge
prefers position A.

In [ ]:
# ============================================================================
# POSITION BIAS FROM NOTEBOOK 10'S PAIRWISE VERDICTS
# ============================================================================
pairwise = custom_judge_scores["pairwise"]
flips = sum(1 for result in pairwise if result["winner"] == "position_flip")
named = [position for result in pairwise for position in result["positions"] if position != "tie"]
first_won = named.count("A")

print(f"🥊 {len(pairwise)} pairs; {flips} position flips ({flips / max(len(pairwise), 1):.0%})")
if named:
    print(f"📐 The reply shown first won {first_won} of {len(named)} named verdicts ({first_won / len(named):.0%})")
print("Reading: near 50% first-wins and few flips means order barely matters. A high flip rate means the "
      "pairwise judge can't separate these replies; count only pairs that win in both orders.")

### 4.2 📏 Verbosity bias

The test: take replies the judge already ruled on, add text that carries no information (a
greeting and a sign-off), and judge them again. The policy-wording rubric says nothing about
length, so the verdict should not move. A fail that turns into a pass after padding is verbosity
bias: the judge reads length as care.

⏱️ 20 judge calls with the Qwen judge (10 passing and 10 failing replies): about a cent in all.

In [ ]:
# ============================================================================
# PAD 10 PASSING AND 10 FAILING REPLIES, JUDGE AGAIN
# ============================================================================
PADDING_BEFORE = "Thank you so much for reaching out to Pinecrest Outdoor Supply, we really appreciate it! "
PADDING_AFTER = " We hope this helps, and please don't hesitate to get in touch if there is anything else at all."

policy_items = [item for item in sample if item["judge"] == "policy_wording"]
padding_set = [item for item in policy_items if item["passed"]][:10] + [item for item in policy_items if not item["passed"]][:10]
policy_judge = rubric_judges(JUDGE_MODEL)["policy_wording"]

for item in padding_set:
    if item["key"] in calibration_verdicts["verbosity"]:
        continue   # judged on an earlier run: not paid for twice
    run = runs_by_id[item["run_id"]]
    padded = PADDING_BEFORE + run["answer"] + PADDING_AFTER
    calibration_verdicts["verbosity"][item["key"]] = judge_passes(policy_judge, test_case_for("policy_wording", run, padded))
save_verdicts()
print(f"✅ {len(padding_set)} padded replies judged")

In [ ]:
# ============================================================================
# DID PADDING CHANGE THE VERDICT?
# ============================================================================
moves = Counter()
for item in padding_set:
    padded_verdict = calibration_verdicts["verbosity"].get(item["key"])
    if padded_verdict is None:
        continue
    before = "pass" if item["passed"] else "fail"
    after = "pass" if padded_verdict else "fail"
    moves[f"{before} → {after}"] += 1
print("📏", dict(moves))
print("Reading: 'fail → pass' is verbosity bias (padding bought a pass). 'pass → fail' is the opposite, a judge "
      "that punishes length the rubric doesn't mention. Both should be 0 or close to it.")

### 4.3 🪞 Self-preference

A model judging its own writing tends to like it. The course keeps the judge (Qwen) a different
family from the app (DeepSeek) for that reason. This check measures what would happen otherwise:
the **app model** judges the sampled items, all of them DeepSeek replies, with the same rubrics. A
higher pass rate than Qwen's on the same items, and a lower TNR against the person, is
self-preference.

⏱️ About 200 judge calls with the DeepSeek app model (the refund tree takes 3–4 calls an item):
well under a dollar.

In [ ]:
# ============================================================================
# THE APP MODEL AS A JUDGE (for this check only)
# ============================================================================
APP = model_config("app")
APP_AS_JUDGE = GPTModel(model=APP["name"], base_url=APP["base_url"], api_key=APP["api_key"],
                        generation_kwargs={"extra_body": APP["extra_body"]}, temperature=0)
APP_AS_JUDGE.model_data.supports_structured_outputs = APP.get("structured_outputs", False)
app_judges = all_judges(APP_AS_JUDGE)

for item in sample:
    if item["key"] in calibration_verdicts["self_preference"]:
        continue
    run = runs_by_id[item["run_id"]]
    calibration_verdicts["self_preference"][item["key"]] = judge_passes(
        app_judges[item["judge"]], test_case_for(item["judge"], run))
save_verdicts()
print(f"✅ {len(sample)} items judged by the app model:", APP["name"])

In [ ]:
# ============================================================================
# PASS RATES: THE APP MODEL ON ITS OWN REPLIES vs THE QWEN JUDGE
# ============================================================================
print(f"{'judge':<20}{'Qwen pass':>10}{'app-model pass':>16}{'app TNR vs person':>19}")
for judge_name in pools:
    items = [item for item in sample if item["judge"] == judge_name
             and calibration_verdicts["self_preference"].get(item["key"]) is not None]
    if not items:
        continue
    qwen_rate = sum(item["passed"] for item in items) / len(items)
    app_rate = sum(calibration_verdicts["self_preference"][item["key"]] for item in items) / len(items)
    pairs = [(calibration_verdicts["self_preference"][item["key"]], human_labels[item["key"]]["label"])
             for item in items if item["key"] in human_labels]
    result = agreement(pairs) if pairs else None
    tnr = fmt(result["tnr"]) if result else "needs labels"
    print(f"{judge_name:<20}{qwen_rate:>10.0%}{app_rate:>16.0%}{tnr:>19}")
print("Reading: an app-model pass rate well above Qwen's, with a lower TNR, is self-preference: the reason the "
      "course never lets the app grade itself.")

---
## 🔁 Part 5: A reference judge

One judge measured against people says how good that judge is. A second, stronger judge on the
same items says more: where Qwen and `gpt-4o` **agree with each other but not with the person**,
the label or the rubric is the suspect; where `gpt-4o` agrees with the person and Qwen doesn't,
Qwen is. Judge-vs-judge kappa and the refund tree against the end state need no human labels, so
this part says something useful even before labelling is done.

`gpt-4o` runs on the calibration sample only, never on the full golden set: it costs about ten
times as much as Qwen.

⏱️ About 240 judge calls with `gpt-4o` on OpenRouter (the refund tree takes 3–4 calls an item):
about $1.

In [ ]:
# ============================================================================
# THE REFERENCE JUDGE: gpt-4o through the same provider as the other roles
# ============================================================================
REFERENCE = model_config("generator")      # OpenRouter; the generator role already uses openai/gpt-4o
REFERENCE_MODEL_NAME = "openai/gpt-4o"
REFERENCE_MODEL = GPTModel(model=REFERENCE_MODEL_NAME, base_url=REFERENCE["base_url"], api_key=REFERENCE["api_key"],
                           generation_kwargs={"extra_body": REFERENCE["extra_body"]}, temperature=0)
REFERENCE_MODEL.model_data.supports_structured_outputs = True   # a provider-prefixed name: say it here
reference_judges = all_judges(REFERENCE_MODEL)

for item in sample:
    if item["key"] in calibration_verdicts["reference"]:
        continue
    run = runs_by_id[item["run_id"]]
    calibration_verdicts["reference"][item["key"]] = judge_passes(
        reference_judges[item["judge"]], test_case_for(item["judge"], run))
save_verdicts()
print(f"✅ {len(sample)} items judged by", REFERENCE_MODEL_NAME)

In [ ]:
# ============================================================================
# JUDGE vs JUDGE (no labels needed)
# ============================================================================
print(f"{'judge':<20}{'n':>4}{'agree':>7}{'kappa':>7}")
for judge_name in pools:
    pairs = [(item["passed"], calibration_verdicts["reference"][item["key"]]) for item in sample
             if item["judge"] == judge_name and calibration_verdicts["reference"].get(item["key"]) is not None]
    result = agreement(pairs)
    if result:
        agree = (result["tp"] + result["tn"]) / result["n"]
        print(f"{judge_name:<20}{result['n']:>4}{agree:>7.0%}{fmt(result['kappa']):>7}")
print("Reading: low kappa between two capable judges means the rubric leaves room for two readings. Fix the "
      "rubric before blaming either model.")

In [ ]:
# ============================================================================
# THE REFUND TREE AGAINST THE END STATE, BOTH JUDGE MODELS (no labels needed)
# ============================================================================
# The end state says whether the *decision* was right. A reply that passes the tree while the decision
# was wrong is a reply that misdescribes what happened: the judge should have caught it.
refund_items = [item for item in sample if item["judge"] == "refund_flow"]
for model_name, verdict_of in [(JUDGE_MODEL_NAME, lambda item: item["passed"]),
                               (REFERENCE_MODEL_NAME, lambda item: calibration_verdicts["reference"].get(item["key"]))]:
    cells = Counter()
    for item in refund_items:
        verdict = verdict_of(item)
        if verdict is None:
            continue
        decision_right = end_state_correct(runs_by_id[item["run_id"]])
        cells[("reply pass" if verdict else "reply fail", "decision right" if decision_right else "decision wrong")] += 1
    print(f"🌳 {model_name}: {dict(cells)}")
print("Reading: 'reply pass, decision wrong' should be 0 for a judge that reads the facts it was given.")

In [ ]:
# ============================================================================
# BOTH JUDGES AGAINST THE PERSON
# ============================================================================
reference_agreement = {}
if not human_labels:
    print("⏸️ Needs human labels (Part 2).")
else:
    print(f"{'judge':<20}{'Qwen kappa':>11}{'gpt-4o kappa':>13}{'gpt-4o TNR':>11}")
    for judge_name in pools:
        pairs = [(calibration_verdicts["reference"][item["key"]], human_labels[item["key"]]["label"])
                 for item in sample if item["judge"] == judge_name and item["key"] in human_labels
                 and calibration_verdicts["reference"].get(item["key"]) is not None]
        result = agreement(pairs)
        if result is None or judge_name not in primary_agreement:
            continue
        reference_agreement[judge_name] = result
        print(f"{judge_name:<20}{fmt(primary_agreement[judge_name]['kappa']):>11}{fmt(result['kappa']):>13}{fmt(result['tnr']):>11}")
    print("Reading: if gpt-4o clears 0.6 where Qwen doesn't, the judge model is the limit; if neither does, the rubric is.")

---
## ✏️ Part 6: Revising a rubric, then measuring again

A judge under the 0.6 bar is not thrown away; its rubric is revised. The loop:

1. Read its disagreements (Part 3) and the judge-vs-judge splits (Part 5). Find the step both
   readings hang on: "plain and to the point" is a typical culprit, because "plain" means
   different things to different readers.
2. Rewrite that one step into something checkable: "at most three sentences, and no sentence that
   repeats the customer's question" instead of "plain and to the point".
3. Edit the rubric in **notebook 10** (the judge's home), and copy the same edit into Part 0.1
   here, so the judge measured is still the judge used.
4. Re-run notebook 10 for fresh verdicts, then this notebook. The sample and the labels stay the
   same (fixed seed, same keys), so only the judge changed between the two kappas.

Never revise a rubric until it agrees with **these** labels and stop: that tunes the judge to one
sample. After a revision, label a fresh sample (change the seed) before trusting the new kappa.

The next cell lists what to revise, from the numbers measured above; it changes nothing itself.

In [ ]:
# ============================================================================
# WHICH JUDGES NEED A REVISION?
# ============================================================================
if not primary_agreement:
    print("⏸️ Needs human labels (Part 2).")
else:
    for judge_name, result in primary_agreement.items():
        if result["kappa"] >= 0.6:
            print(f"✅ {judge_name:<20} kappa {result['kappa']:.2f}: trusted as is")
        else:
            misses = "lets failures through" if (result["tnr"] or 0) < (result["tpr"] or 0) else "fails good replies"
            print(f"✏️ {judge_name:<20} kappa {result['kappa']:.2f}: revise the rubric; it mostly {misses}")

---
## 💾 Part 7: The judge trust table

One row per judge model and metric: how many items a person labelled, the three agreement
numbers, and `trusted`, which is `kappa ≥ 0.6`. Notebook 13's release gate reads this file and
counts a judge's scores only where the primary judge's row says `trusted: true`.

The file is written **only from human labels**. Without them, no file is written, and the gate
treats every judge as untrusted, which is the right default for a judge nobody has checked.

In [ ]:
# ============================================================================
# SAVE → data/judge_trust.json
# ============================================================================
if not primary_agreement:
    print("⏸️ No human labels, so no trust table: every judge stays untrusted until a person checks it.")
else:
    trust_rows = []
    for role, model_name, results in [("primary", JUDGE_MODEL_NAME, primary_agreement),
                                      ("reference", REFERENCE_MODEL_NAME, reference_agreement)]:
        for judge_name, result in results.items():
            trust_rows.append({"judge": role, "metric": judge_name, "judge_model": model_name,
                               "n_labelled": result["n"], "kappa": round(result["kappa"], 3),
                               "tpr": None if result["tpr"] is None else round(result["tpr"], 3),
                               "tnr": None if result["tnr"] is None else round(result["tnr"], 3),
                               "trusted": result["kappa"] >= 0.6})
    with open(os.path.join("data", "judge_trust.json"), "w", encoding="utf-8") as f:
        json.dump(trust_rows, f, indent=2)
    trusted = [row["metric"] for row in trust_rows if row["judge"] == "primary" and row["trusted"]]
    print(f"✅ {len(trust_rows)} rows → data/judge_trust.json; primary judges trusted: {trusted or 'none'}")

---
## 📝 Summary

In this notebook, we learned:

### 1. Sampling for calibration
- **Stratify by verdict**: up to half fails, so the judge's ability to catch a failure is actually tested
- **Fix the seed**: the same items come back, so labels stay attached to them

### 2. Human labels without anchoring
- **Blind labelling**: the person sees the rule and the evidence, never the judge's verdict or reason
- **Two routes, one file**: a LangSmith annotation queue or a JSON file, both writing `data/human_labels.json`
- **No labels, no trust**: every number that needs a person says so instead of guessing

### 3. Agreement in plain Python
- **Kappa, not accuracy**: agreement corrected for chance; 0.6 is the bar for gating
- **TNR matters most for a gate**: it is the share of bad replies the judge catches

### 4. Bias checks
- **Position**: a winner counts only if it wins in both orders
- **Verbosity**: padding should never turn a fail into a pass
- **Self-preference**: the app model grading its own replies is why judge and app stay different families

### 5. A reference judge and the trust table
- **Two judges, one person**: where the judges agree with each other but not the person, suspect the rubric or the label
- **`data/judge_trust.json`**: one row per judge model and metric; the gate counts only trusted primary judges

### Next Steps
- **`12_statistics_cost_and_experiments.ipynb`**: per-slice reports with bootstrap confidence
  intervals, paired comparisons of two app versions, cost and latency from LangSmith, and the golden
  set run as LangSmith experiments